# Notebook 04: Estimated Marginal Means (Adjusted Means) - Table 7

### Study: Generative AI in EFL Academic Writing, Self-Regulated Learning (SRL), and AI Literacy
This notebook computes the Estimated Marginal Means (EMMs / Adjusted Means) and Standard Errors for the Experimental and Control groups, holding pre-test covariates at their overall sample means.

The outputs replicate **Table 7** of the research document:
- Adjusted Means ($	ext{Mean}_{	ext{adj}}$)
- Standard Errors ($SE$)
- 95% Confidence Intervals
- Mean Difference and Cohen's $d$ / Effect Size metrics


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load dataset
df = pd.read_csv('/mnt/data/Hypothetical_EFL_AI_Study_Data.csv')
print(f"Dataset loaded: {df.shape[0]} records.")


## 1. Computation Function for Adjusted Means (EMMs)
To calculate adjusted means:
1. Fit the ANCOVA model $Y = eta_0 + eta_1 X_{	ext{pre}} + eta_2 	ext{Group}$.
2. Create prediction frames for Experimental and Control groups fixing $X_{	ext{pre}} = ar{X}_{	ext{pre}}$ (the overall mean).
3. Compute predicted values and standard errors of prediction.


In [ ]:
def compute_adjusted_means(data, post_col, pre_col, group_col='Group'):
    # Fit regression model
    formula = f"{post_col} ~ {pre_col} + C({group_col})"
    model = smf.ols(formula, data=data).fit()
    
    # Grand mean of the covariate
    cov_mean = data[pre_col].mean()
    groups = sorted(data[group_col].unique())
    
    results = []
    for g in groups:
        pred_df = pd.DataFrame({
            pre_col: [cov_mean],
            group_col: [g]
        })
        pred = model.get_prediction(pred_df)
        pred_summary = pred.summary_frame(alpha=0.05)
        adj_mean = pred_summary['mean'].values[0]
        se = pred_summary['mean_se'].values[0]
        ci_lower = pred_summary['mean_ci_lower'].values[0]
        ci_upper = pred_summary['mean_ci_upper'].values[0]
        
        # Raw post mean and SD for comparison
        raw_mean = data[data[group_col] == g][post_col].mean()
        raw_sd = data[data[group_col] == g][post_col].std()
        n = (data[group_col] == g).sum()
        
        results.append({
            'Group': g,
            'N': n,
            'Raw Mean': raw_mean,
            'Raw SD': raw_sd,
            'Adjusted Mean': adj_mean,
            'SE': se,
            '95% CI Lower': ci_lower,
            '95% CI Upper': ci_upper
        })
    
    res_df = pd.DataFrame(results)
    return res_df, model

print("Helper function defined.")


## 2. Generating Table 7 across All 3 Domains
Now we compute adjusted means for Writing, SRL, and AI Literacy.


In [ ]:
domains = [
    ('Writing Performance', 'Writing_Post', 'Writing_Pre'),
    ('Self-Regulated Learning (SRL)', 'SRL_Post', 'SRL_Pre'),
    ('AI Literacy', 'AI_Post', 'AI_Pre')
]

table7_rows = []

for domain_name, post_var, pre_var in domains:
    res_df, model = compute_adjusted_means(df, post_var, pre_var)
    exp_row = res_df[res_df['Group'] == 'Experimental'].iloc[0]
    ctrl_row = res_df[res_df['Group'] == 'Control'].iloc[0]
    
    diff = exp_row['Adjusted Mean'] - ctrl_row['Adjusted Mean']
    
    # Calculate pooled residual SD
    mse = model.mse_resid
    pooled_sd = np.sqrt(mse)
    cohen_d = diff / pooled_sd
    
    table7_rows.append({
        'Outcome Domain': domain_name,
        'Group': 'Experimental',
        'Raw Mean (SD)': f"{exp_row['Raw Mean']:.2f} ({exp_row['Raw SD']:.2f})",
        'Adjusted Mean': f"{exp_row['Adjusted Mean']:.2f}",
        'SE': f"{exp_row['SE']:.2f}",
        '95% CI': f"[{exp_row['95% CI Lower']:.2f}, {exp_row['95% CI Upper']:.2f}]",
        'Adjusted Diff (Exp - Ctrl)': f"{diff:.2f}",
        "Cohen's d (adj)": f"{cohen_d:.2f}"
    })
    table7_rows.append({
        'Outcome Domain': domain_name,
        'Group': 'Control',
        'Raw Mean (SD)': f"{ctrl_row['Raw Mean']:.2f} ({ctrl_row['Raw SD']:.2f})",
        'Adjusted Mean': f"{ctrl_row['Adjusted Mean']:.2f}",
        'SE': f"{ctrl_row['SE']:.2f}",
        '95% CI': f"[{ctrl_row['95% CI Lower']:.2f}, {ctrl_row['95% CI Upper']:.2f}]",
        'Adjusted Diff (Exp - Ctrl)': "-",
        "Cohen's d (adj)": "-"
    })

table7_df = pd.DataFrame(table7_rows)
print("=== TABLE 7: Adjusted Post-Test Means, Standard Errors, and Differences ===")
table7_df


## 3. Visual Comparison: Raw vs Adjusted Post-Test Means


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (domain_name, post_var, pre_var) in zip(axes, domains):
    res_df, _ = compute_adjusted_means(df, post_var, pre_var)
    x = np.arange(len(res_df))
    width = 0.35
    
    ax.bar(x - width/2, res_df['Raw Mean'], width, label='Raw Mean', color='#7eb0d5', alpha=0.9)
    ax.bar(x + width/2, res_df['Adjusted Mean'], width, yerr=res_df['SE'], capsize=5, label='Adjusted Mean (ANCOVA)', color='#2b5c8f', alpha=0.9)
    
    ax.set_xticks(x)
    ax.set_xticklabels(res_df['Group'], fontsize=11, fontweight='bold')
    ax.set_title(domain_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
